# Stage 3 - Real-Time Streaming (Kafka + Spark Structured Streaming)

**Project:** MediaWave Unified Analytics Platform  
**Notebook Purpose:** Guided implementation and grading notebook for Stage 3  
**Focus:** Kafka topic design, event parsing, concurrent viewer tracking, trending spike detection, and poor experience alerts

This notebook is written in a step-by-step style so the reader can follow the business purpose, the technical setup, and the expected output from each section.

## Assignment objective

In this stage, the streaming pipeline reads MediaWave user activity events from Kafka and applies Spark Structured Streaming logic to support two operational goals:

1. Monitor **concurrent viewers per title** in near real time.
2. Trigger alerts for:
   - sessions with **more than 3 buffering events**, and
   - titles whose viewership increases by **more than 200% in a 15-minute window**.

This notebook is organized like a graded class notebook: setup first, schema second, parsing third, then one streaming task per section.

In [ ]:
# Spark session setup
# This follows the direct and explicit coding style used in class notebooks.

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MediaWave-Stage3-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark application id:", spark.sparkContext.applicationId)

## Parameters

These values are kept in one place so the instructor can quickly see what should be edited before execution.

- `TOPIC_NAME` should match the Kafka topic created for MediaWave user activity.
- `KAFKA_BOOTSTRAP` points to the Kafka broker inside the Docker network.
- `CHECKPOINT_ROOT` is optional but recommended for stable reruns.

In [ ]:
# Editable parameters

TOPIC_NAME = "mediawave-user-activity"
KAFKA_BOOTSTRAP = "kafka:9092"
CHECKPOINT_ROOT = "/tmp/mediawave_stage3_checkpoints"

print("Topic:", TOPIC_NAME)
print("Kafka bootstrap server:", KAFKA_BOOTSTRAP)
print("Checkpoint root:", CHECKPOINT_ROOT)

## Step 1 - Read the raw Kafka stream

This section creates the raw streaming DataFrame directly from Kafka. At this point, the Kafka `value` field is still binary, so the next step will parse it into columns.

In [ ]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

raw_stream.printSchema()

## Step 2 - Define the event schema

The event schema below reflects the Stage 3 requirements and adds practical fields that make streaming analysis easier:

- `session_id` is used for buffering alerts.
- `action` identifies play, pause, browse, skip, search, and buffering events.
- `event_ts` is the event-time field used for window-based calculations.
- `title` is included so the output can be read directly by a grader without joining another table.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

activity_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("action", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("device", StringType(), True)
])

## Step 3 - Parse the Kafka JSON payload

This section converts the binary Kafka message into structured columns. It also converts the event timestamp into a Spark timestamp column so window functions can be used later.

In [ ]:
from pyspark.sql.functions import col, from_json, to_timestamp

parsed_stream = (
    raw_stream
    .selectExpr("CAST(key AS STRING) AS message_key", "CAST(value AS STRING) AS json_string")
    .select(from_json(col("json_string"), activity_schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("event_ts")))
)

parsed_stream.printSchema()

## Step 4 - Quick stream preview

Before running alerts, it is helpful to preview the parsed event stream. This section is useful for grading because it shows whether the Kafka records are reaching Spark and whether the schema matches the actual event payload.

In [ ]:
preview_query = (
    parsed_stream
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after a short preview when running in class.
# preview_query.stop()

## Step 5 - Concurrent viewers by title

A simple operational approximation for concurrent viewers is used here:

- count active `play` events by title,
- evaluate them in short windows,
- and display the current viewer count for each title.

If the project later includes explicit stop or session-end events, this logic can be refined further.

In [ ]:
from pyspark.sql.functions import window, count

play_events = parsed_stream.filter(col("action") == "play")

concurrent_viewers = (
    play_events
    .withWatermark("event_time", "15 minutes")
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("content_id"),
        col("title")
    )
    .agg(count("user_id").alias("concurrent_viewers"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("content_id"),
        col("title"),
        col("concurrent_viewers")
    )
)

concurrent_viewers_query = (
    concurrent_viewers
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after viewing results.
# concurrent_viewers_query.stop()

## Step 6 - Poor experience alert: buffering events > 3 per session

This section looks for sessions that exceed the poor experience threshold. The assignment rule is straightforward: a session should be flagged when buffering occurs more than 3 times.

This logic assumes buffering records appear in the same topic as activity events with `action = 'buffering'`.

In [ ]:
from pyspark.sql.functions import sum, when

buffering_events = parsed_stream.withColumn(
    "buffer_flag",
    when(col("action") == "buffering", 1).otherwise(0)
)

session_buffer_counts = (
    buffering_events
    .withWatermark("event_time", "30 minutes")
    .groupBy(col("session_id"))
    .agg(sum("buffer_flag").alias("buffering_count"))
)

poor_experience_alerts = session_buffer_counts.filter(col("buffering_count") > 3)

poor_experience_query = (
    poor_experience_alerts
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after viewing alerts.
# poor_experience_query.stop()

## Step 7 - Trending spike alert: more than 200% growth in 15 minutes

This section estimates trending behavior by comparing title-level play counts across 15-minute windows.

For grading clarity, the notebook computes:
- a current 15-minute viewer count,
- a lagged baseline count,
- a percentage growth value,
- and an alert when growth is greater than 200%.

Because exact production logic can vary, this notebook emphasizes transparency over optimization.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

viewer_windows = (
    play_events
    .withWatermark("event_time", "30 minutes")
    .groupBy(
        window(col("event_time"), "15 minutes", "15 minutes"),
        col("content_id"),
        col("title")
    )
    .agg(count("user_id").alias("viewer_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("content_id"),
        col("title"),
        col("viewer_count")
    )
)

# NOTE:
# The lag calculation below is easier to explain in a notebook than in a pure streaming sink.
# Depending on runtime behavior, some environments may prefer writing viewer_windows to memory or Delta first.

window_spec = Window.partitionBy("content_id").orderBy("window_start")

viewer_growth = (
    viewer_windows
    .withColumn("previous_viewer_count", lag("viewer_count").over(window_spec))
    .withColumn(
        "growth_ratio",
        (col("viewer_count") - col("previous_viewer_count")) / col("previous_viewer_count")
    )
)

trending_alerts = viewer_growth.filter(
    (col("previous_viewer_count").isNotNull()) &
    (col("previous_viewer_count") > 0) &
    (col("growth_ratio") > 2.0)
)

## Step 8 - Display trending alerts

If the environment supports the lag-based calculation above in the chosen sink, the results can be printed to the console. If not, the grader can still inspect `viewer_windows` and the notebook can be adapted to write to a memory table first.

In [ ]:
# In some Spark environments, streaming queries with lag/window combinations may need a staged approach.
# This cell is included to show the intended output path.

# trending_query = (
#     trending_alerts
#     .writeStream
#     .outputMode("update")
#     .format("console")
#     .option("truncate", "false")
#     .start()
# )
#
# trending_query.stop()

## Step 9 - Suggested grading checkpoints

A grader reviewing this notebook should be able to confirm the following:

1. Kafka is read successfully from the MediaWave topic.
2. JSON messages are parsed into structured columns.
3. Play events are separated from other actions.
4. Concurrent viewers are grouped by title and event-time window.
5. Poor experience sessions are flagged when buffering count is greater than 3.
6. Trending logic compares a title's current 15-minute activity against an earlier window.

If needed, screenshots of console output can be captured from the preview, concurrent viewer stream, and poor experience alert stream.

## Notes for final project submission

Before final submission, update these items if needed:

- Confirm the exact Kafka topic name used by the group.
- Confirm whether buffering events are in the same topic or a separate topic.
- Replace any placeholder field names so they match the actual Stage 1 output.
- If required by the professor, save alert output to Kafka, a file sink, or a database sink instead of the console.